<a href="https://colab.research.google.com/github/kaviyasri2405/machine-learning/blob/main/ex_17.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
# Generate synthetic data for demonstration
np.random.seed(42)

num_samples = 1000

data = {
    'RAM_GB': np.random.randint(2, 16, num_samples),
    'Storage_GB': np.random.choice([32, 64, 128, 256, 512], num_samples),
    'Screen_Size_Inches': np.round(np.random.uniform(5.0, 7.0, num_samples), 1),
    'Camera_MP': np.random.randint(12, 108, num_samples),
    'Battery_mAh': np.random.randint(3000, 6000, num_samples),
    'Brand': np.random.choice(['Samsung', 'Apple', 'Xiaomi', 'OnePlus', 'Google'], num_samples)
}
df = pd.DataFrame(data)

# Create a target variable (price) with some noise
df['Price'] = (
    df['RAM_GB'] * 50
    + df['Storage_GB'] * 2
    + df['Screen_Size_Inches'] * 30
    + df['Camera_MP'] * 5
    + df['Battery_mAh'] * 0.1
    + (df['Brand'] == 'Apple') * 300
    + (df['Brand'] == 'Samsung') * 100
    + np.random.normal(0, 100, num_samples)
)
df['Price'] = df['Price'].round(2)

In [3]:
# Define features (X) and target (y)
X = df.drop('Price', axis=1)
y = df['Price']

# Identify categorical and numerical features
categorical_features = ['Brand']
numerical_features = [col for col in X.columns if col not in categorical_features]

# Create a preprocessor using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough' # Keep numerical features as they are
)

# Create a pipeline with preprocessing and RandomForestRegressor
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
model_pipeline.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model_pipeline.predict(X_test)

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error: {mae:.2f}")
print(f"R-squared: {r2:.2f}")

# Example prediction for a new mobile phone
new_mobile = pd.DataFrame([[8, 128, 6.2, 48, 4500, 'Xiaomi']],
                          columns=['RAM_GB', 'Storage_GB', 'Screen_Size_Inches', 'Camera_MP', 'Battery_mAh', 'Brand'])
predicted_price = model_pipeline.predict(new_mobile)
print(f"Predicted Price for new mobile: ${predicted_price[0]:.2f}")

Mean Absolute Error: 108.88
R-squared: 0.92
Predicted Price for new mobile: $1537.63
